# Generate Dataset 1 (XLSX) for journal submission

Exports `df_SC_Ent` (filtered dataset: pLDDT ≥ 70, no transmembrane/secretory proteins) to an Excel file with:
- **Dataset1** sheet: selected columns, renamed (`entangled` → `NCLE`, `clustered_entangled_residues` → `NCLE region`) and ordered for publication.
- **Info** sheet: description of each field.

Residue lists are written as comma-separated strings (empty cell = no residues).

In [1]:
import pandas as pd
from openpyxl.styles import Alignment, Font

IN_FILE = "../data/SC_Ent_no_transmembrane_secretory.pkl"
OUT_FILE = "../data/Dataset1.xlsx"

df_SC_Ent = pd.read_pickle(IN_FILE)
print(df_SC_Ent.shape)

(1887, 12)


In [2]:
RENAME = {
    "entangled": "NCLE",
    "clustered_entangled_residues": "NCLE region",
}
COLUMN_ORDER = [
    "SGDID", "Uniprot", "SC", "length_AF", "NCLE",
    "asphericity", "fraction_idp", "NCLE region", "Cutsite_residues",
]
LIST_COLUMNS = ["NCLE region", "Cutsite_residues"]

df_out = df_SC_Ent.rename(columns=RENAME)[COLUMN_ORDER].reset_index(drop=True)

# Excel cannot store Python lists: serialize residue lists as sorted, comma-separated strings
def list_to_str(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return ", ".join(str(int(r)) for r in sorted(x))

for col in LIST_COLUMNS:
    df_out[col] = df_out[col].apply(list_to_str)

# Excel cell limit is 32,767 characters
assert df_out[LIST_COLUMNS].map(len).max().max() < 32767, "Residue string exceeds Excel cell limit"

print(df_out.shape)
df_out.head()

(1887, 9)


,SGDID,Uniprot,SC,length_AF,NCLE,asphericity,fraction_idp,NCLE region,Cutsite_residues
0,S000002468,Q12298,0,539,1,0.139968,0.011132,"27, 31, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44...",
1,S000002490,P38961,0,392,1,0.108081,0.295918,"181, 193, 266, 267, 268, 290, 292, 293, 294, 2...",
2,S000002198,P12945,0,854,0,0.074009,0.069087,,
3,S000003242,P53204,0,395,1,0.102152,0.293671,"1, 2, 3, 4, 5, 6, 7, 8, 9, 15, 16, 17, 18, 19,...",
4,S000005430,P33895,0,451,1,0.855840,0.414634,"4, 5, 6, 7, 8, 9, 10, 11, 16, 97, 98, 99, 100,...",


In [3]:
info = pd.DataFrame(
    [
        ("SGDID", "Systematic Gene Identifier from the Saccharomyces Genome Database for each yeast gene."),
        ("Uniprot", "UniProt accession number corresponding to the protein sequence."),
        ("SC", "Binary indicator of age associated structural change detected by LiP MS analysis, where 1 indicates significant structural change and 0 indicates no significant structural change."),
        ("length_AF", "Protein sequence length in residues based on the AlphaFold model."),
        ("NCLE", "Binary indicator denoting whether the protein contains at least one native non covalent lasso entanglement."),
        ("asphericity", "Shape anisotropy metric calculated from the protein\u2019s radius of gyration tensor, where higher values indicate more elongated, less spherical structures."),
        ("fraction_idp", "Fraction of intrinsically disordered residues in the protein, estimated from MobiDB."),
        ("NCLE region", "List of residues classified as part of natively entangled regions, including sequence neighbors and spatial contact neighbors of the entanglement residues."),
        ("Cutsite_residues", "Residues identified as significant proteolytic alteration cut sites in the LiP MS experiment."),
    ],
    columns=["Term", "Annotation"],
)
assert list(info["Term"]) == COLUMN_ORDER
info

,Term,Annotation
0,SGDID,Systematic Gene Identifier from the Saccharomy...
1,Uniprot,UniProt accession number corresponding to the ...
2,SC,Binary indicator of age associated structural ...
3,length_AF,Protein sequence length in residues based on t...
4,NCLE,Binary indicator denoting whether the protein ...
5,asphericity,Shape anisotropy metric calculated from the pr...
6,fraction_idp,Fraction of intrinsically disordered residues ...
7,NCLE region,List of residues classified as part of nativel...
8,Cutsite_residues,Residues identified as significant proteolytic...


In [4]:
with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    df_out.to_excel(writer, sheet_name="Dataset1", index=False)
    info.to_excel(writer, sheet_name="Info", index=False)

    # Dataset sheet: bold centered header, freeze header row, reasonable widths
    ws = writer.sheets["Dataset1"]
    widths = {"SGDID": 13, "Uniprot": 11, "SC": 6, "length_AF": 11, "NCLE": 7,
              "asphericity": 12, "fraction_idp": 13, "NCLE region": 50, "Cutsite_residues": 30}
    for i, col in enumerate(COLUMN_ORDER, start=1):
        cell = ws.cell(row=1, column=i)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")
        ws.column_dimensions[cell.column_letter].width = widths[col]
    ws.freeze_panes = "A2"

    # Info sheet: bold header, wrapped annotation text
    ws = writer.sheets["Info"]
    ws.column_dimensions["A"].width = 18
    ws.column_dimensions["B"].width = 100
    for cell in ws[1]:
        cell.font = Font(bold=True)
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")

print(f"Wrote {OUT_FILE}")

Wrote ../data/Dataset1.xlsx


In [5]:
# Round-trip check
check = pd.read_excel(OUT_FILE, sheet_name=None)
for name, sheet in check.items():
    print(name, sheet.shape, list(sheet.columns))
assert len(check["Dataset1"]) == len(df_SC_Ent)

Dataset1 (1887, 9) ['SGDID', 'Uniprot', 'SC', 'length_AF', 'NCLE', 'asphericity', 'fraction_idp', 'NCLE region', 'Cutsite_residues']
Info (9, 2) ['Term', 'Annotation']
